# Experiment 7: Bagging, Boosting & Classification with Data Augmentation

**Objective**: Implement Bagging and Boosting ensemble algorithms, and study the effect of data augmentation on classification performance.

**Dataset**: Breast Cancer Wisconsin (from scikit-learn)

**Algorithms**:
- Bagging Classifier & Random Forest
- AdaBoost & Gradient Boosting
- Classification with SMOTE and Gaussian-noise augmentation

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    BaggingClassifier,
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score, roc_curve
)
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("Libraries imported successfully.")

## 2. Load and Explore Dataset

In [ ]:
# Load Breast Cancer dataset
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target
feature_names = cancer.feature_names
target_names = cancer.target_names   # ['malignant', 'benign']

df = pd.DataFrame(X, columns=feature_names)
df['target'] = y

print(f"Dataset Shape: {X.shape}")
print(f"Features: {len(feature_names)}")
print(f"Classes : {list(target_names)}")
print(f"\nClass Distribution:")
print(pd.Series(y).value_counts().rename({0: 'Malignant', 1: 'Benign'}))
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
print("Statistical Summary:")
df.describe()

## 3. Data Preprocessing

In [ ]:
print("Missing Values:", df.isnull().sum().sum())

In [ ]:
# Train-test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Training samples : {X_train.shape[0]}")
print(f"Test samples     : {X_test.shape[0]}")
print("Feature scaling applied.")

## 4. Helper — evaluate any model

In [ ]:
def evaluate(model, X_tr, y_tr, X_te, y_te, name='Model'):
    """Train, predict, and return a metrics dict."""
    model.fit(X_tr, y_tr)
    y_pred  = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]
    m = {
        'accuracy' : accuracy_score(y_te, y_pred),
        'precision': precision_score(y_te, y_pred),
        'recall'   : recall_score(y_te, y_pred),
        'f1'       : f1_score(y_te, y_pred),
        'auc'      : roc_auc_score(y_te, y_proba)
    }
    print(f"{name}")
    print(f"  Accuracy : {m['accuracy']:.4f}   Precision: {m['precision']:.4f}")
    print(f"  Recall   : {m['recall']:.4f}   F1-Score : {m['f1']:.4f}")
    print(f"  AUC-ROC  : {m['auc']:.4f}")
    return m, y_pred, y_proba

## 5. Baseline — Single Decision Tree

In [ ]:
dt_metrics, y_pred_dt, y_proba_dt = evaluate(
    DecisionTreeClassifier(random_state=42),
    X_train_scaled, y_train, X_test_scaled, y_test,
    name='Baseline — Decision Tree'
)

---
# Part A — Bagging Algorithms

**Bagging (Bootstrap Aggregating)** reduces **variance** by:
1. Drawing bootstrap samples from training data
2. Training independent models in parallel
3. Aggregating via majority vote

## 6. Bagging Classifier

In [ ]:
bag_metrics, y_pred_bag, y_proba_bag = evaluate(
    BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=42),
        n_estimators=100, max_samples=0.8, max_features=0.8,
        bootstrap=True, random_state=42, n_jobs=-1
    ),
    X_train_scaled, y_train, X_test_scaled, y_test,
    name='Bagging Classifier (100 trees)'
)

## 7. Random Forest

In [ ]:
rf_clf = RandomForestClassifier(
    n_estimators=100, max_features='sqrt',
    random_state=42, n_jobs=-1
)
rf_metrics, y_pred_rf, y_proba_rf = evaluate(
    rf_clf,
    X_train_scaled, y_train, X_test_scaled, y_test,
    name='Random Forest (100 trees)'
)

In [ ]:
# Feature Importance
fi = pd.DataFrame({'feature': feature_names,
                   'importance': rf_clf.feature_importances_}
     ).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 7))
plt.barh(fi['feature'][:15], fi['importance'][:15], color='steelblue')
plt.xlabel('Importance'); plt.ylabel('Feature')
plt.title('Top 15 Feature Importances (Random Forest)', fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('../figures/exp07_rf_importance.png', dpi=150, bbox_inches='tight')
plt.show()

---
# Part B — Boosting Algorithms

**Boosting** reduces **bias** by:
1. Training weak learners sequentially
2. Re-weighting misclassified samples
3. Combining with weighted voting

## 8. AdaBoost

In [ ]:
ada_metrics, y_pred_ada, y_proba_ada = evaluate(
    AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1),
        n_estimators=100, learning_rate=1.0,
        algorithm='SAMME', random_state=42
    ),
    X_train_scaled, y_train, X_test_scaled, y_test,
    name='AdaBoost (100 stumps)'
)

## 9. Gradient Boosting

In [ ]:
gb_metrics, y_pred_gb, y_proba_gb = evaluate(
    GradientBoostingClassifier(
        n_estimators=100, learning_rate=0.1,
        max_depth=3, subsample=0.8, random_state=42
    ),
    X_train_scaled, y_train, X_test_scaled, y_test,
    name='Gradient Boosting'
)

## 10. Ensemble Comparison

In [ ]:
results = {
    'Decision Tree'     : dt_metrics,
    'Bagging'           : bag_metrics,
    'Random Forest'     : rf_metrics,
    'AdaBoost'          : ada_metrics,
    'Gradient Boosting' : gb_metrics
}

comp_df = pd.DataFrame(results).T.round(4)
print("\n" + "="*75)
print("ENSEMBLE COMPARISON TABLE")
print("="*75)
print(comp_df.to_string())

In [ ]:
# Bar chart: all metrics side-by-side
metrics_list = ['accuracy', 'precision', 'recall', 'f1', 'auc']
models = list(results.keys())
x = np.arange(len(models))
w = 0.15
colors = ['#2ecc71', '#3498db', '#9b59b6', '#e74c3c', '#f39c12']

fig, ax = plt.subplots(figsize=(14, 6))
for i, met in enumerate(metrics_list):
    vals = [results[m][met] for m in models]
    ax.bar(x + i*w, vals, w, label=met.upper(), color=colors[i])

ax.set_xlabel('Model'); ax.set_ylabel('Score')
ax.set_title('Ensemble Methods — Metric Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x + w*2); ax.set_xticklabels(models, rotation=15, ha='right')
ax.legend(loc='lower right'); ax.set_ylim(0.85, 1.02)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/exp07_ensemble_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ROC Curves
fig, ax = plt.subplots(figsize=(9, 7))

for y_proba, name, color in [
    (y_proba_dt,  'Decision Tree',     'gray'),
    (y_proba_bag, 'Bagging',           'blue'),
    (y_proba_rf,  'Random Forest',     'green'),
    (y_proba_ada, 'AdaBoost',          'orange'),
    (y_proba_gb,  'Gradient Boosting',  'red')
]:
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc_val = roc_auc_score(y_test, y_proba)
    ax.plot(fpr, tpr, color=color, lw=2,
            label=f'{name} (AUC={auc_val:.4f})')

ax.plot([0,1],[0,1],'k--', lw=1, label='Random')
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC Curves — Ensemble Methods', fontsize=14, fontweight='bold')
ax.legend(loc='lower right'); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/exp07_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for idx, (y_pred, name) in enumerate([
    (y_pred_dt,  'Decision Tree'),
    (y_pred_bag, 'Bagging'),
    (y_pred_rf,  'Random Forest'),
    (y_pred_ada, 'AdaBoost'),
    (y_pred_gb,  'Gradient Boosting')
]):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=target_names, yticklabels=target_names)
    axes[idx].set_title(name, fontweight='bold', fontsize=10)
    axes[idx].set_xlabel('Predicted'); axes[idx].set_ylabel('Actual')

plt.tight_layout()
plt.savefig('../figures/exp07_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Effect of Number of Estimators

In [ ]:
n_est_range = [10, 25, 50, 75, 100, 150, 200]
scores = {k: [] for k in ['Bagging','Random Forest','AdaBoost','Gradient Boosting']}

for n in n_est_range:
    for cls_name, cls in [
        ('Bagging',           BaggingClassifier(n_estimators=n, random_state=42, n_jobs=-1)),
        ('Random Forest',     RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)),
        ('AdaBoost',          AdaBoostClassifier(n_estimators=n, random_state=42)),
        ('Gradient Boosting', GradientBoostingClassifier(n_estimators=n, random_state=42))
    ]:
        cls.fit(X_train_scaled, y_train)
        scores[cls_name].append(cls.score(X_test_scaled, y_test))

plt.figure(figsize=(10, 5))
styles = [('b-o','Bagging'),('g-s','Random Forest'),('orange','AdaBoost'),('r-d','Gradient Boosting')]
for (st, nm) in styles:
    if nm == 'AdaBoost':
        plt.plot(n_est_range, scores[nm], color='orange', marker='^', lw=2, ms=7, label=nm)
    else:
        plt.plot(n_est_range, scores[nm], st, lw=2, ms=7, label=nm)

plt.xlabel('Number of Estimators'); plt.ylabel('Test Accuracy')
plt.title('Accuracy vs n_estimators', fontsize=14, fontweight='bold')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/exp07_n_estimators.png', dpi=150, bbox_inches='tight')
plt.show()

---
# Part C — Classification with Data Augmentation

We now **artificially create an imbalanced version** of the dataset and show
how **data augmentation** (SMOTE and Gaussian noise injection) can recover
classification performance.

## 12. Create an Imbalanced Dataset

In [ ]:
# Down-sample malignant class to only 40 samples → 40 vs 357 (≈ 10 : 1)
from sklearn.utils import resample

df_full = pd.DataFrame(X, columns=feature_names)
df_full['target'] = y

df_mal = df_full[df_full.target == 0]   # malignant
df_ben = df_full[df_full.target == 1]   # benign

# Keep only 40 malignant samples
df_mal_down = resample(df_mal, replace=False, n_samples=40, random_state=42)
df_imb = pd.concat([df_mal_down, df_ben])

X_imb = df_imb.drop('target', axis=1).values
y_imb = df_imb['target'].values

print("Imbalanced Dataset:")
print(pd.Series(y_imb).value_counts().rename({0: 'Malignant', 1: 'Benign'}))
print(f"Imbalance ratio: 1 : {sum(y_imb==1)//sum(y_imb==0)}")

In [ ]:
# Split imbalanced data
X_imb_tr, X_imb_te, y_imb_tr, y_imb_te = train_test_split(
    X_imb, y_imb, test_size=0.2, random_state=42, stratify=y_imb
)

scaler_imb = StandardScaler()
X_imb_tr_s = scaler_imb.fit_transform(X_imb_tr)
X_imb_te_s = scaler_imb.transform(X_imb_te)

print(f"Train — Malignant: {sum(y_imb_tr==0)}, Benign: {sum(y_imb_tr==1)}")
print(f"Test  — Malignant: {sum(y_imb_te==0)}, Benign: {sum(y_imb_te==1)}")

## 13. Baseline on Imbalanced Data (no augmentation)

In [ ]:
print("--- WITHOUT Data Augmentation (imbalanced) ---\n")

imb_results_no_aug = {}
imb_preds_no_aug = {}

for name, clf in [
    ('Decision Tree',     DecisionTreeClassifier(random_state=42)),
    ('Random Forest',     RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ('AdaBoost',          AdaBoostClassifier(n_estimators=100, random_state=42)),
    ('Gradient Boosting', GradientBoostingClassifier(n_estimators=100, random_state=42))
]:
    m, yp, _ = evaluate(clf, X_imb_tr_s, y_imb_tr, X_imb_te_s, y_imb_te, name=name)
    imb_results_no_aug[name] = m
    imb_preds_no_aug[name] = yp
    print()

## 14. Data Augmentation — SMOTE

**SMOTE** (Synthetic Minority Oversampling TEchnique) creates synthetic
samples by interpolating between existing minority-class neighbours.

In [ ]:
# SMOTE implementation from scratch (no imblearn needed)
from sklearn.neighbors import NearestNeighbors

def smote(X_minority, y_minority_label, n_synthetic, k=5, random_state=42):
    """Generate synthetic samples via SMOTE."""
    rng = np.random.RandomState(random_state)
    nn = NearestNeighbors(n_neighbors=k+1).fit(X_minority)
    distances, indices = nn.kneighbors(X_minority)

    synthetic_X = []
    for _ in range(n_synthetic):
        idx = rng.randint(0, len(X_minority))
        neighbour = indices[idx][rng.randint(1, k+1)]   # skip self
        diff = X_minority[neighbour] - X_minority[idx]
        gap  = rng.uniform(0, 1)
        synthetic_X.append(X_minority[idx] + gap * diff)

    synthetic_X = np.array(synthetic_X)
    synthetic_y = np.full(n_synthetic, y_minority_label)
    return synthetic_X, synthetic_y

# ------- apply SMOTE -------
minority_mask = y_imb_tr == 0
X_min = X_imb_tr_s[minority_mask]
n_to_generate = sum(y_imb_tr == 1) - sum(y_imb_tr == 0)  # balance the classes

X_syn, y_syn = smote(X_min, 0, n_to_generate)

X_smote = np.vstack([X_imb_tr_s, X_syn])
y_smote = np.concatenate([y_imb_tr, y_syn])

print(f"Before SMOTE — Malignant: {sum(y_imb_tr==0)}, Benign: {sum(y_imb_tr==1)}")
print(f"After  SMOTE — Malignant: {sum(y_smote==0)}, Benign: {sum(y_smote==1)}")

In [ ]:
print("--- WITH SMOTE Augmentation ---\n")
imb_results_smote = {}
imb_preds_smote = {}

for name, clf in [
    ('Decision Tree',     DecisionTreeClassifier(random_state=42)),
    ('Random Forest',     RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ('AdaBoost',          AdaBoostClassifier(n_estimators=100, random_state=42)),
    ('Gradient Boosting', GradientBoostingClassifier(n_estimators=100, random_state=42))
]:
    m, yp, _ = evaluate(clf, X_smote, y_smote, X_imb_te_s, y_imb_te, name=name)
    imb_results_smote[name] = m
    imb_preds_smote[name] = yp
    print()

## 15. Data Augmentation — Gaussian Noise Injection

Add random Gaussian noise to minority-class samples to expand
the training set.

In [ ]:
def gaussian_augment(X_minority, y_label, n_synthetic, noise_std=0.1, random_state=42):
    """Create synthetic samples by adding Gaussian noise."""
    rng = np.random.RandomState(random_state)
    indices = rng.randint(0, len(X_minority), n_synthetic)
    noise   = rng.normal(0, noise_std, (n_synthetic, X_minority.shape[1]))
    X_aug   = X_minority[indices] + noise
    y_aug   = np.full(n_synthetic, y_label)
    return X_aug, y_aug

X_gaug, y_gaug = gaussian_augment(X_min, 0, n_to_generate)

X_gauss = np.vstack([X_imb_tr_s, X_gaug])
y_gauss = np.concatenate([y_imb_tr, y_gaug])

print(f"Before Augmentation — Malignant: {sum(y_imb_tr==0)}, Benign: {sum(y_imb_tr==1)}")
print(f"After  Gaussian Aug — Malignant: {sum(y_gauss==0)}, Benign: {sum(y_gauss==1)}")

In [ ]:
print("--- WITH Gaussian Noise Augmentation ---\n")
imb_results_gauss = {}
imb_preds_gauss = {}

for name, clf in [
    ('Decision Tree',     DecisionTreeClassifier(random_state=42)),
    ('Random Forest',     RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ('AdaBoost',          AdaBoostClassifier(n_estimators=100, random_state=42)),
    ('Gradient Boosting', GradientBoostingClassifier(n_estimators=100, random_state=42))
]:
    m, yp, _ = evaluate(clf, X_gauss, y_gauss, X_imb_te_s, y_imb_te, name=name)
    imb_results_gauss[name] = m
    imb_preds_gauss[name] = yp
    print()

## 16. Augmentation Impact — Comparison

In [ ]:
# Compare F1 (most relevant for imbalanced data)
aug_comp = pd.DataFrame({
    'No Augmentation': {m: imb_results_no_aug[m]['f1'] for m in imb_results_no_aug},
    'SMOTE':           {m: imb_results_smote[m]['f1'] for m in imb_results_smote},
    'Gaussian Noise':  {m: imb_results_gauss[m]['f1'] for m in imb_results_gauss}
}).round(4)

print("\n" + "="*65)
print("F1-SCORE COMPARISON — Effect of Data Augmentation")
print("="*65)
print(aug_comp.to_string())

In [ ]:
# Recall comparison (critical for medical diagnosis — catching malignant cases)
recall_comp = pd.DataFrame({
    'No Augmentation': {m: imb_results_no_aug[m]['recall'] for m in imb_results_no_aug},
    'SMOTE':           {m: imb_results_smote[m]['recall'] for m in imb_results_smote},
    'Gaussian Noise':  {m: imb_results_gauss[m]['recall'] for m in imb_results_gauss}
}).round(4)

print("\n" + "="*65)
print("RECALL COMPARISON — Effect of Data Augmentation")
print("="*65)
print(recall_comp.to_string())

In [ ]:
# Grouped bar chart: F1 score comparison
model_names = list(imb_results_no_aug.keys())
x = np.arange(len(model_names))
w = 0.25

f1_no  = [imb_results_no_aug[m]['f1'] for m in model_names]
f1_sm  = [imb_results_smote[m]['f1']  for m in model_names]
f1_ga  = [imb_results_gauss[m]['f1']  for m in model_names]

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(x - w, f1_no, w, label='No Augmentation', color='#e74c3c')
ax.bar(x,     f1_sm, w, label='SMOTE',           color='#2ecc71')
ax.bar(x + w, f1_ga, w, label='Gaussian Noise',  color='#3498db')

ax.set_xlabel('Model'); ax.set_ylabel('F1 Score')
ax.set_title('F1 Score: Impact of Data Augmentation on Imbalanced Data',
             fontsize=13, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(model_names, rotation=15, ha='right')
ax.legend(); ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 1.1)

# add value labels
for bars in ax.containers:
    ax.bar_label(bars, fmt='%.2f', fontsize=8)

plt.tight_layout()
plt.savefig('../figures/exp07_augmentation_f1.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Visualise the synthetic samples (first 2 PCA components)
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca_orig = pca.fit_transform(X_imb_tr_s)
X_pca_smot = pca.transform(X_syn)
X_pca_gaus = pca.transform(X_gaug)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Original
axes[0].scatter(X_pca_orig[y_imb_tr==1, 0], X_pca_orig[y_imb_tr==1, 1],
                c='steelblue', alpha=0.6, label='Benign', s=30)
axes[0].scatter(X_pca_orig[y_imb_tr==0, 0], X_pca_orig[y_imb_tr==0, 1],
                c='coral', alpha=0.8, label='Malignant', s=50, edgecolors='k', linewidths=0.5)
axes[0].set_title('Original Imbalanced Data', fontweight='bold')
axes[0].legend()

# After SMOTE
axes[1].scatter(X_pca_orig[y_imb_tr==1, 0], X_pca_orig[y_imb_tr==1, 1],
                c='steelblue', alpha=0.6, label='Benign', s=30)
axes[1].scatter(X_pca_orig[y_imb_tr==0, 0], X_pca_orig[y_imb_tr==0, 1],
                c='coral', alpha=0.8, label='Malignant (orig)', s=50, edgecolors='k', linewidths=0.5)
axes[1].scatter(X_pca_smot[:, 0], X_pca_smot[:, 1],
                c='lime', alpha=0.5, label='SMOTE synthetic', s=30, marker='x')
axes[1].set_title('After SMOTE', fontweight='bold')
axes[1].legend()

# After Gaussian Noise
axes[2].scatter(X_pca_orig[y_imb_tr==1, 0], X_pca_orig[y_imb_tr==1, 1],
                c='steelblue', alpha=0.6, label='Benign', s=30)
axes[2].scatter(X_pca_orig[y_imb_tr==0, 0], X_pca_orig[y_imb_tr==0, 1],
                c='coral', alpha=0.8, label='Malignant (orig)', s=50, edgecolors='k', linewidths=0.5)
axes[2].scatter(X_pca_gaus[:, 0], X_pca_gaus[:, 1],
                c='gold', alpha=0.5, label='Gaussian synthetic', s=30, marker='x')
axes[2].set_title('After Gaussian Noise', fontweight='bold')
axes[2].legend()

for ax in axes:
    ax.set_xlabel('PC 1'); ax.set_ylabel('PC 2')
    ax.grid(True, alpha=0.2)

plt.suptitle('PCA Visualization of Data Augmentation', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../figures/exp07_augmentation_pca.png', dpi=150, bbox_inches='tight')
plt.show()

## 17. Cross-Validation

In [ ]:
print("5-Fold Cross-Validation (balanced original dataset):")
print("="*60)

cv_models = {
    'Decision Tree':     DecisionTreeClassifier(random_state=42),
    'Bagging':           BaggingClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Random Forest':     RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'AdaBoost':          AdaBoostClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

cv_results = {}
for name, model in cv_models.items():
    sc = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy')
    cv_results[name] = sc
    print(f"{name:20} — Mean: {sc.mean():.4f}  (±{sc.std()*2:.4f})")

In [ ]:
# Box plot
plt.figure(figsize=(10, 5))
pd.DataFrame(cv_results).boxplot()
plt.ylabel('Accuracy'); plt.xticks(rotation=15)
plt.title('5-Fold CV Accuracy', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/exp07_cv_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

## 18. Summary and Conclusions

In [ ]:
best = max(results.items(), key=lambda x: x[1]['accuracy'])

print("\n" + "="*75)
print("EXPERIMENT 7 SUMMARY")
print("="*75)

print("\n📊  PART A & B — ENSEMBLE RESULTS (balanced dataset):")
print(comp_df.to_string())
print(f"\n🏆  Best Ensemble Model: {best[0]}  (Accuracy={best[1]['accuracy']:.4f})")

In [ ]:
print("\n📊  PART C — DATA AUGMENTATION RESULTS (imbalanced dataset):")
print("\nF1 Scores:")
print(aug_comp.to_string())

print("\nRecall (malignant detection rate):")
print(recall_comp.to_string())

In [ ]:
print("\n" + "="*75)
print("KEY OBSERVATIONS")
print("="*75)

print("""
1. BAGGING (reduces variance)
   ✓ Trains independent models on bootstrap samples in parallel
   ✓ Random Forest adds random feature selection → further de-correlates trees
   ✓ Robust to outliers; less prone to overfitting

2. BOOSTING (reduces bias)
   ✓ Trains weak learners sequentially; each corrects predecessors' errors
   ✓ AdaBoost re-weights samples; Gradient Boosting fits residuals
   ✓ Often achieves highest accuracy but needs careful tuning

3. DATA AUGMENTATION
   ✓ SMOTE interpolates between minority neighbours → realistic synthetic points
   ✓ Gaussian noise adds jittered copies → quick expansion of minority set
   ✓ Both techniques improve recall & F1 on imbalanced data
   ✓ Augmented ensembles outperform vanilla ensembles on skewed classes
""")

print("BAGGING  vs  BOOSTING")
print("-"*50)
print("Training    : Parallel          │ Sequential")
print("Goal        : ↓ Variance        │ ↓ Bias")
print("Weights     : Equal samples     │ Adaptive weights")
print("Overfitting : Less prone        │ More prone")
print("Outliers    : Robust            │ Sensitive")
print("Examples    : RF, Bagging       │ AdaBoost, GBM")

In [ ]:
print("\n" + "="*75)
print("Experiment 7 completed successfully!")
print("="*75)